# [센서 실습 18] 가속도 센서 – MPU-6050 (가속도)

6축 관성 센서 **MPU-6050**의 가속도 레지스터를 I2C로 직접 읽어 3축 가속도를 구하고, **자유낙하 → 충격 → 자세 변화** 패턴으로 낙상을 감지합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
MEMS 가속도 센서 안에는 스프링에 매달린 아주 작은 **추**가 있어, 가속되면 추가 밀려나며 정전용량이 변합니다.
- 가만히 있어도 **중력 때문에 1g(9.8m/s²)** 가 측정됨 → 어느 쪽이 아래인지 알 수 있음
- 레지스터 `0x3B`부터 6바이트: X·Y·Z 각 16비트 **부호 있는 정수**(2의 보수)
- ±2g 범위에서 **1g = 16384**

| 상황 | 가속도 크기 $\sqrt{a_x^2 + a_y^2 + a_z^2}$ |
|---|---|
| 정지 | 약 1g |
| 걷기 | 0.8 ~ 1.3g 흔들림 |
| 자유낙하 | 0g 가까이 |
| 바닥 충돌 | 순간 2 ~ 4g 이상 |

### 부품 사양
| 항목 | 내용 |
|---|---|
| 가속도 범위 | ±2 / 4 / 8 / 16 g 선택 |
| 분해능 | 16비트 (±2g: 16384 LSB/g) |
| 자이로 | ±250 ~ 2000 °/s (19번 실습) |
| 전원 | 3.3V (모듈은 3.3~5V) |
| 통신 | I2C 주소 0x68 |

## 2. 라즈베리파이 연결 방법
**준비물**: MPU-6050 모듈(GY-521), 점퍼선 4개

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MPU-6050 VCC | 3.3V (1번 핀) |  |
| MPU-6050 GND | GND (6번 핀) |  |
| MPU-6050 SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| MPU-6050 SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 (주소 0x68) |



**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install smbus2
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
레지스터에서 읽은 6바이트를 부호 있는 16비트 정수로 바꾸고, 감도로 나누어 m/s²로 변환합니다.

In [ ]:
ACCEL_SCALE = 16384.0      # ±2g 범위: 1g = 16384
G = 9.80665

def to_int16(hi, lo):
    v = (hi << 8) | lo
    return v - 65536 if v >= 32768 else v           # 2의 보수

def bytes_to_accel(block):
    """[XH, XL, YH, YL, ZH, ZL] → (ax, ay, az) m/s²"""
    return tuple(round(to_int16(block[i], block[i + 1]) / ACCEL_SCALE * G, 3) for i in (0, 2, 4))

print(bytes_to_accel([0x00, 0x00, 0x00, 0x00, 0x40, 0x00]))   # Z = 16384 → 1g

### 4-2. 실제 센서 클래스
절전 모드를 해제(`0x6B` ← 0)한 뒤, `0x3B`부터 6바이트를 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealMPU6050Accel:
    PWR_MGMT_1, ACCEL_XOUT_H = 0x6B, 0x3B

    def __init__(self, address=0x68, bus=1):
        from smbus2 import SMBus
        self.bus, self.addr = SMBus(bus), address
        self.bus.write_byte_data(self.addr, self.PWR_MGMT_1, 0)        # 절전 해제

    def read(self):
        block = self.bus.read_i2c_block_data(self.addr, self.ACCEL_XOUT_H, 6)
        ax, ay, az = bytes_to_accel(block)
        return {"ax": ax, "ay": ay, "az": az}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 구간 | 가정한 동작 | 가속도 |
|---|---|---|
| 0 ~ 2초 | 서 있음 (센서를 허리에 착용) | Z축 1g |
| 2 ~ 6초 | 걷기 (약 2걸음/초) | Z축 1g ± 0.25g, X축 ± 0.1g |
| 6.0 ~ 6.35초 | 넘어지며 떨어짐 | 거의 0g (자유낙하) |
| 6.35 ~ 6.45초 | 바닥 충돌 | 순간 약 3.5g (X축은 ±2g 범위를 넘어 **2g에서 잘림 = 포화**) |
| 6.5초 이후 | 바닥에 누움 | 중력이 X축으로 이동 (자세 변화) |

가상 센서는 가속도를 **±32767 범위의 16비트 정수 6바이트**로 바꿔 실제와 같은 변환식에 넣습니다.

In [ ]:
class VirtualMPU6050Accel:
    def __init__(self, dt=0.02, seed=18):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def _true_g(self):
        t = self.t
        if t < 2:        return np.array([0.0, 0.0, 1.0])
        if t < 6:        return np.array([0.1 * np.sin(2 * np.pi * 2 * t), 0.0, 1 + 0.25 * np.sin(2 * np.pi * 2 * t)])
        if t < 6.35:     return np.array([0.05, 0.05, 0.08])
        if t < 6.45:     return np.array([2.8, 0.5, 2.0])
        return np.array([1.0, 0.05, 0.05])                       # 누운 자세

    def read(self):
        g = self._true_g() + self.rng.normal(0, 0.02, 3)
        raw = np.clip(np.round(g * ACCEL_SCALE), -32768, 32767).astype(int)
        block = []
        for v in raw:
            v &= 0xFFFF
            block += [v >> 8, v & 0xFF]                          # 실제 레지스터와 같은 6바이트
        self.t += self.dt
        ax, ay, az = bytes_to_accel(block)
        return {"ax": ax, "ay": ay, "az": az}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualMPU6050Accel(dt=0.02)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
50Hz(0.02초 간격)로 10초 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 500               # 측정 횟수
INTERVAL = 0.02               # 측정 간격(초)

sensor = RealMPU6050Accel() if USE_REAL_SENSOR else VirtualMPU6050Accel(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
낙상은 다음 3단계가 **순서대로** 일어날 때로 판정합니다.
1. **자유낙하**: 크기 < 0.4g
2. 그 후 0.5초 안에 **충격**: 크기 > 2.5g
3. 충격 1초 뒤 **자세 변화**: Z축 성분이 0.5g 미만 (누워 있음)

In [ ]:
df["mag_g"] = np.sqrt(df.ax ** 2 + df.ay ** 2 + df.az ** 2) / G
free = df[df["mag_g"] < 0.4]
fall = None
for _, f in free.iterrows():
    hit = df[(df.time_s > f.time_s) & (df.time_s <= f.time_s + 0.5) & (df.mag_g > 2.5)]
    if len(hit):
        h = hit.iloc[0]
        after = df[(df.time_s >= h.time_s + 1.0)].head(5)
        if len(after) and (after.az / G).abs().mean() < 0.5:
            fall = (f.time_s, h.time_s, h.mag_g); break
if fall:
    print(f"🚨 낙상 감지: 자유낙하 {fall[0]:.2f}초 → 충격 {fall[1]:.2f}초 (최대 {fall[2]:.1f}g) → 누운 자세 확인")
    print("   → 보호자에게 긴급 알림 전송")
print(f"걷는 동안 최대 {df[(df.time_s > 2) & (df.time_s < 6)].mag_g.max():.2f}g → 걷기는 낙상으로 판정되지 않음")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df.time_s, df.ax / G, "k-", lw=1, label="X"); ax[0].plot(df.time_s, df.az / G, "k--", lw=1, label="Z")
ax[0].plot(df.time_s, df.ay / G, color="gray", lw=0.8, label="Y"); ax[0].set_ylabel("g"); ax[0].legend()
ax[0].set_title("MPU-6050 가속도 – 낙상 감지"); ax[0].grid(alpha=0.3)
ax[1].plot(df.time_s, df.mag_g, "k-", lw=1.2, label="가속도 크기")
ax[1].axhline(0.4, color="gray", ls=":", label="자유낙하 0.4g"); ax[1].axhline(2.5, color="gray", ls="--", label="충격 2.5g")
ax[1].set_ylabel("g"); ax[1].set_xlabel("시간 (초)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "가속도계센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "MPU-6050 (가속도)",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/accel", payload)
# client.disconnect()

## 직접 해 보기
- 충격 기준을 1.3g로 낮추면 걷기도 낙상으로 잘못 판정되는지 확인해 보세요.
- 범위를 ±8g로 바꾸면 `ACCEL_SCALE`은 얼마가 되어야 할까요? (데이터시트: 4096)
- 정지 상태의 가속도로 기울기 각도를 계산해 보세요. (각도 = atan2(ax, az))